In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install transformers torch pandas scipy

In [15]:
import pandas as pd
import numpy as np
from transformers import pipeline

print("Loading sentiment model (downloads ~500MB on first run) ...")

classifier = pipeline(
    task            = "sentiment-analysis",
    model           = "cardiffnlp/twitter-roberta-base-sentiment-latest",
    tokenizer       = "cardiffnlp/twitter-roberta-base-sentiment-latest",
    top_k           = 3,       # Explicitly request top 3 labels
    truncation      = True,
    max_length      = 512,
)

print("✓ Model loaded\n")

def classify_sentiment(text: str) -> float:
    """
    Runs the RoBERTa classifier and calculates a -1 to 1 Sentiment Score
    based on the exact confidence probabilities:
      +1  → very enthusiastic / positive
      0   → neutral / objective
      -1  → very fatigued / negative / disgusted
    """
    try:
        raw_out = classifier(str(text).strip())

        # The pipeline output with top_k > 1 is typically [[{...}, {...}, {...}]]
        # We need to extract the inner list of dictionaries
        if isinstance(raw_out, list) and len(raw_out) > 0 and isinstance(raw_out[0], list):
            preds = raw_out[0]
        else:
            print(f"  ⚠️  Unexpected output format from classifier: {raw_out} → defaulting to 0.0")
            return 0.0

        # Map model labels to sentiment names
        label_map = {
            "LABEL_0": "negative",
            "LABEL_1": "neutral",
            "LABEL_2": "positive",
        }

        # Convert list of dicts to a single probability map: {'negative': 0.8, 'neutral': 0.15, 'positive': 0.05}
        probs = {label_map.get(p["label"], p["label"].lower()): p["score"] for p in preds}

        # Calculate weighted -1 to 1 sentiment score
        sentiment_score = (probs.get("positive", 0.0) * 1) + \
                          (probs.get("neutral", 0.0) * 0) + \
                          (probs.get("negative", 0.0) * -1)

        return round(sentiment_score, 2)
    except Exception as e:
        print(f"  ⚠️  Error classifying text: {e} → defaulting to 0.0")
        return 0.0

# ── CELL 5: Quick sanity test ─────────────────────────────────
print("=== Sanity test (Checking -1 to 1 variance) ===")
tests = [
    "I am so sick of seeing Coachella content everywhere, totally overexposed.", # Should be negative (near -1)
    "The weather was nice at the festival.",                                     # Should be neutral (near 0)
    "Coachella was absolutely amazing this year, best lineup ever!",             # Should be positive (near 1)
]
for text in tests:
    score = classify_sentiment(text)
    print(f"  Score: {score:+05.2f} | '{text[:60]}...'")
print()

# ── CELL 6: Load data ─────────────────────────────────────────
df_reddit = pd.read_csv("/content/reddit_comments.csv")
df_search = pd.read_csv("/content/search_data.csv")

df_reddit["Date"] = pd.to_datetime(df_reddit["Date"]).dt.strftime("%Y-%m-%d")
df_search["Date"] = pd.to_datetime(df_search["Date"]).dt.strftime("%Y-%m-%d")

print(f"Reddit comments : {len(df_reddit)} rows")
print(f"Search data     : {len(df_search)} rows\n")

# ── CELL 7: Score every comment ───────────────────────────────
print(f"Scoring {len(df_reddit)} comments (no API calls — running locally) ...")

scores = []

for i, row in df_reddit.iterrows():
    score = classify_sentiment(row["Comment_Text"])
    scores.append(score)

    if (i + 1) % 25 == 0 or (i + 1) == len(df_reddit):
        current_mean = sum(scores) / len(scores)
        print(f"  [{i+1}/{len(df_reddit)}] Average sentiment score so far: {current_mean:.2f}")

df_reddit["Sentiment_Score"] = scores
print(f"\n✓ Scoring complete.")

# ── CELL 8: Calculate daily Sentiment Score ────────────────────
#
# Sentiment_Score = daily mean of the -1 to 1 scale
#   ~1  → very enthusiastic / positive
#   ~0  → neutral / objective
#   ~-1 → very fatigued / negative / disgusted

df_sentiment = (
    df_reddit
    .groupby(["Date", "Keyword"])["Sentiment_Score"]
    .mean()
    .round(2)
    .reset_index()
    .rename(columns={"Sentiment_Score": "Sentiment_Score"})
)

print(f"\nSentiment scores for {len(df_sentiment)} Date×Keyword pairs")
print(df_sentiment.groupby("Keyword")["Sentiment_Score"].describe().round(2).to_string())

# ── CELL 9: Merge with search data & FIX BLANKS ──────────────
df_final = df_search.merge(
    df_sentiment,
    on  = ["Date", "Keyword"],
    how = "left",
)
df_final = df_final.sort_values(["Keyword", "Date"]).reset_index(drop=True)

# CRITICAL FIX: Interpolate to replace NaNs instead of forward/backward fill
# This ensures every single date has a score and introduces more variability for the UI line chart.
df_final['Sentiment_Score'] = df_final.groupby('Keyword')['Sentiment_Score'].transform(lambda x: x.interpolate(method='linear').ffill().bfill())
df_final['Sentiment_Score'] = df_final['Sentiment_Score'].round(2)

print(f"\n── Final dataset ───────────────────────────────────────────")
print(f"Rows            : {len(df_final)}")
print(f"Columns         : {df_final.columns.tolist()}")
print(f"Sentiment non-null: {df_final['Sentiment_Score'].notna().sum()} / {len(df_final)} (Should be 100% now)")
print(f"\nSample (First 8 rows of Final Dataset):")
print(df_final.head(8).to_string(index=False))

# ── CELL 10: Save outputs ─────────────────────────────────────
df_final.to_csv("precomputed_cases.csv", index=False, encoding="utf-8-sig")
print(f"\n✅ Saved 'precomputed_cases.csv' (Ready for Streamlit!)")

df_reddit.to_csv("reddit_comments_scored.csv", index=False, encoding="utf-8-sig")
print(f"✅ Saved 'reddit_comments_scored.csv'")

Loading sentiment model (downloads ~500MB on first run) ...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 
roberta.pooler.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Model loaded

=== Sanity test (Checking -1 to 1 variance) ===
  Score: -0.92 | 'I am so sick of seeing Coachella content everywhere, totally...'
  Score: +0.97 | 'The weather was nice at the festival....'
  Score: +0.98 | 'Coachella was absolutely amazing this year, best lineup ever...'

Reddit comments : 200 rows
Search data     : 122 rows

Scoring 200 comments (no API calls — running locally) ...
  [25/200] Average sentiment score so far: 0.10
  [50/200] Average sentiment score so far: 0.19
  [75/200] Average sentiment score so far: 0.21
  [100/200] Average sentiment score so far: 0.21
  [125/200] Average sentiment score so far: 0.20
  [150/200] Average sentiment score so far: 0.18
  [175/200] Average sentiment score so far: 0.16
  [200/200] Average sentiment score so far: 0.14

✓ Scoring complete.

Sentiment scores for 29 Date×Keyword pairs
           count  mean   std   min   25%   50%   75%   max
Keyword                                                   
Alo Yoga    25.0  0.19  